# Circular Imports — The Complete Picture

Circular imports are one of the most misunderstood Python topics.  
Most developers avoid them by instinct without knowing exactly why they fail — or when they work.

In [ ]:
import sys
from pathlib import Path

_demo = Path.cwd() / 'demo_packages'
if not _demo.exists():
    _demo = Path.cwd() / 'modules_and_imports' / 'demo_packages'
assert _demo.exists(), f'demo_packages not found — expected: {_demo}'

if str(_demo) not in sys.path:
    sys.path.insert(0, str(_demo))
print(f'demo_packages on path: {sys.path[0]}')

## 1. The partial module problem

Circular imports exist when two modules import each other:

```
broken_a.py: from circular_demo.broken_b import func_b
broken_b.py: from circular_demo.broken_a import func_a
```

Python's Step 4 from the import lifecycle (register before execute) is what causes the failure:

```
1. Import triggered: import circular_demo.broken_a
2. Python creates an empty stub: sys.modules['circular_demo.broken_a'] = <empty Module>
3. Python starts executing broken_a.py
4. Line 1 of broken_a.py:  from circular_demo.broken_b import func_b
5. Python now imports broken_b
6. Python creates a stub: sys.modules['circular_demo.broken_b'] = <empty Module>
7. Python starts executing broken_b.py
8. Line 1 of broken_b.py:  from circular_demo.broken_a import func_a
9. broken_a IS in sys.modules — Python returns the stub from step 2
10. The stub exists but broken_a.py hasn't finished yet — func_a is NOT defined on it
11. ImportError: cannot import name 'func_a' from 'circular_demo.broken_a'
```

The failure is always the same: **a symbol is requested from a module that hasn't finished executing yet.**

In [ ]:
# Clean any previous state
for key in list(sys.modules.keys()):
    if 'circular_demo' in key:
        del sys.modules[key]
print('sys.modules cleared of circular_demo entries')

In [ ]:
import importlib

try:
    # This will fail — see broken_a.py for the full trace
    importlib.import_module('circular_demo.broken_a')
except ImportError as e:
    print(f'ImportError: {e}')
except Exception as e:
    print(f'{type(e).__name__}: {e}')

In [ ]:
# After the failed import, inspect what's left in sys.modules
# Python leaves PARTIAL module objects in the cache
remnants = {k: sys.modules[k] for k in sys.modules if 'broken' in k}
for name, mod in remnants.items():
    if mod is not None:
        names_defined = [n for n in vars(mod) if not n.startswith('_')]
        print(f'{name}')
        print(f'  type: {type(mod).__name__}')
        print(f'  names defined so far: {names_defined}')
    else:
        print(f'{name} = None (failed import placeholder)')

In [ ]:
# Clean up before next demos
for key in list(sys.modules.keys()):
    if 'circular_demo' in key:
        del sys.modules[key]
print('Cleaned up')

## 2. Fix 1 — deferred import (move import inside the function)

The simplest fix: **move the import from module level into the function body**.  
The import runs at call time, not at load time. By call time, both modules are fully initialized.

```python
# ❌ broken — import at module level
from circular_demo.broken_b import func_b   # runs when broken_a.py loads

# ✅ fixed — import deferred into function body
def func_a():
    from circular_demo.fixed_b import func_b   # runs when func_a() is called
    return func_b()
```

**Trade-offs:**
- Cost: the `import` statement runs on every call (but is fast — Step 1 hits the cache immediately)
- Readability: imports at the top of the file are the convention — breaking this is a code smell
- When to use: only for circular import resolution; not as a general practice

In [ ]:
# fixed_a imports fixed_b INSIDE func_a() — no top-level cross-import
# Watch: only [fixed_a] and [fixed_b] print appear, and they're clean

import circular_demo.fixed_a as fixed_a
print()
result = fixed_a.func_a()
print(f'fixed_a.func_a() = {result!r}')

## 3. Fix 2 — import the module object, not a symbol

`from X import name` requires `name` to exist on the module **at import time**.  
`import X as mod` only requires the module **object** to exist — not any specific names on it.

Since Python registers the stub in `sys.modules` before executing the file (Step 4),  
a module object is always available — even mid-initialization.

```python
# ❌ fails: requires func_a to be defined NOW
from circular_demo.broken_a import func_a

# ✅ works: only requires the module object to exist (it's already a stub)
import circular_demo.module_ref_a as mod_a
# mod_a.func_a() is resolved at CALL TIME — by then mod_a is fully initialized
```

In [ ]:
# module_ref_a uses: import circular_demo.module_ref_b as mod_b
# It calls mod_b.func_b() inside a function — resolved at call time

for key in list(sys.modules.keys()):
    if 'module_ref' in key:
        del sys.modules[key]

import circular_demo.module_ref_a as mod_a
print()
result = mod_a.func_a()
print(f'mod_a.func_a() = {result!r}')

## 4. Fix 3 — restructure (the right long-term answer)

Deferred imports are a workaround. The real fix is to eliminate the circular dependency.

**Pattern: extract shared code into a third module**

```
Before (circular):
  a.py  ←→  b.py   (each imports from the other)

After (clean):
  common.py  ←  a.py
  common.py  ←  b.py
  (a and b share code via common; no cycle)
```

**Real-world Django example:**

```python
# models.py ← signals.py is a common circular pattern
# models.py imports signal handlers
# signal handlers import model classes

# Solution 1 (deferred): connect signals inside AppConfig.ready()
# class MyAppConfig(AppConfig):
#     def ready(self):
#         import myapp.signals   # deferred — models are fully loaded by now

# Solution 2 (restructure): move shared constants/types to models/base.py
# signals.py imports from models.base — no need to import from models directly
```

**Rule of thumb:** if you're adding deferred imports to escape a circular dependency,  
that's a signal to examine whether the two modules really belong in the same layer.

## 5. Quick reference — which form works in circular scenarios

| Import form | Works in circular? | Why |
|---|---|---|
| `import X` | **Yes** | Only needs the module object (stub is enough) |
| `import X as m` | **Yes** | Same — attribute access deferred to call time |
| `from X import name` | **Depends** | Needs `name` to be defined NOW — fails if X isn't done executing |
| `from X import name` (deferred) | **Yes** | Moved inside function — runs after both modules are initialized |

**The interview answer:**  
*"Circular imports fail when `from X import name` is used at module level and `X` hasn't finished executing yet. The fix is either to move the import inside the function, import the module object instead of the symbol, or restructure to eliminate the cycle."*